In [1]:
import yt
import unyt
from unyt import unyt_quantity, unyt_array, km, s, Mpc, kg, m, Msun, kpc
from unyt import Gyr, Mpc, km, s
from unyt.physical_constants import G as G_unyt, kboltz, mp as mp_unyt  # G, k_B, m_p

import numpy as np
import matplotlib.pyplot as plt
import caesar
import pynbody as pnb
import pynbody.units as u
import pynbody.array as pna

import os
from astro_constants import NA_no_units, NA, kB_no_units, kB, mH, mp_no_units, mp, mu_e, mu, G_no_units, G
#from astro_constants import NA_no_units, NA, kB_no_units, kB, mH, mp_no_units, mp, mu_e, mu, G_no_units,

# import miniball
code = 'Simba-C'
target_halo = 2919
snap_file = '/scratch/b/babul/aspadawe/snapshots/Hyenasc_L1_zooms/halo_2919/snap_halo_2919_151.hdf5'
# snap_file = '/scratch/b/babul/aspadawe/snapshots/HyenasC/L0/halo_3224/snapshot_050.hdf5'
snap = yt.load(snap_file)#, unit_system='cgs')
s = pnb.load(snap_file)
id_filepath = "/home/b/babul/stlock/entropy_profiles/finished/ids_formatted.txt" # containing halo data/ids


yt : [INFO     ] 2025-03-26 13:36:32,531 ComovingIntegrationOn does not exist, falling back to OmegaLambda
yt : [INFO     ] 2025-03-26 13:36:32,545 Calculating time from 1.000e+00 to be 4.375e+17 seconds
yt : [INFO     ] 2025-03-26 13:36:32,546 Assuming length units are in kpc/h (comoving)
yt : [INFO     ] 2025-03-26 13:36:32,638 Parameters: current_time              = 4.3748565942961094e+17 s
yt : [INFO     ] 2025-03-26 13:36:32,639 Parameters: domain_dimensions         = [1 1 1]
yt : [INFO     ] 2025-03-26 13:36:32,640 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2025-03-26 13:36:32,640 Parameters: domain_right_edge         = [200000. 200000. 200000.]
yt : [INFO     ] 2025-03-26 13:36:32,641 Parameters: cosmological_simulation   = True
yt : [INFO     ] 2025-03-26 13:36:32,641 Parameters: current_redshift          = 4.440892098500626e-16
yt : [INFO     ] 2025-03-26 13:36:32,641 Parameters: omega_lambda              = 0.7
yt : [INFO     ] 2025-03-26 13:36:32,642 

In [2]:
## Metal indices
if code.lower() == 'simba':
    # Simba
    metals_idx = {'Z':0, 'He': 1, 'C': 2, 'N': 3, 'O': 4, 'Ne': 5, 'Mg': 6, 'Si': 7, 'S': 8, 'Ca': 9, 'Fe': 10}
    metals_idx_xigrm = {'H':0, 'He': 1, 'C': 2, 'N': 3, 'O': 4, 'Ne': 5, 'Mg': 6, 'Si': 7, 'S': 8, 'Ca': 9, 'Fe': 10}
    elems = ['H', 'He', 'C', 'N', 'O', 'Ne', 'Mg', 'Si', 'S', 'Ca', 'Fe']
elif code.lower() == 'simba-c':
    # Simba-C
    metals_idx = {'Z':0, 'H':1, 'He': 2, 'Li': 3, 'Be': 4, 'B': 5, 'C': 6, 'N': 7, 'O': 8, 'F': 9, 'Ne': 10, 'Na': 11, 'Mg': 12, 'Al':13,
                  'Si': 14, 'P': 15, 'S': 16, 'Cl': 17, 'Ar': 18, 'K': 19,'Ca': 20, 'Sc': 21, 'Ti': 22, 'V': 23, 'Cr': 24,
                  'Mn': 25, 'Fe': 26, 'Co': 27, 'Ni': 28, 'Cu': 29, 'Zn': 30}
    metals_idx_xigrm = {'H':0, 'He': 1, 'Li': 2, 'Be': 3, 'B': 4, 'C': 5, 'N': 6, 'O': 7, 'F': 8, 'Ne': 9, 'Na': 10, 'Mg': 11, 
                        'Al':12, 'Si': 13, 'P': 14, 'S': 15, 'Cl': 16, 'Ar': 17, 'K': 18,'Ca': 19, 'Sc': 20, 'Ti': 21, 'V': 22, 'Cr': 23,
                      'Mn': 24, 'Fe': 25, 'Co': 26, 'Ni': 27, 'Cu': 28, 'Zn': 29}
    elems=['H', 'He', 'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne', 'Na', 'Mg', 'Al', 'Si',
           'P', 'S', 'Cl', 'Ar', 'K','Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn']#, 'Ga', 'Ge']\n",
else:
    raise SystemExit('Code is not Simba or Simba-C')


In [3]:
# Get data, and some functions

data = snap.all_data()

def gas_entropy(field, data):
    return data["gas", "kT"] * data["gas", "El_number_density"] ** (-2.0/3.0)

def gas_entropy2(field, data):
    return data["gas", "kT"] * data["PartType0", "El_number_density"] ** (-2.0/3.0)

snap.add_field(
    ("gas", "gas_entropy"),
    units="keV*cm**2",
    function=gas_entropy,
    sampling_type="cell",
)

snap.add_field(
    ("PartType0", "gas_entropy"),
    units="keV*cm**2",
    function=gas_entropy2,
    sampling_type="particle",
    force_override=True
)

gas_entropy = data['gas', 'entropy']


def get_coordinates_by_id(id_filepath, target_id):
    with open(id_filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6:
                continue  # skip malformed lines
            
            try:
                obj_id = int(parts[2])
                if obj_id == target_id:
                    # parts[4] contains the coordinate string: (x,y,z)
                    coord_str = parts[4].strip('()')
                    x, y, z = map(float, coord_str.split(','))
                    return (x, y, z)
            except (ValueError, IndexError):
                continue  # skip malformed lines
    
    return None  # if ID not found



yt : [INFO     ] 2025-03-26 13:36:33,192 Allocating for 5.057e+07 particles
Loading particle index: 100%|██████████| 90/90 [00:00<00:00, 3161.54it/s]
/home/b/babul/stlock/.virtualenvs/entropy_venv_12/lib/python3.11/site-packages/unyt/array.py:1949: RuntimeWarning: invalid value encountered in power
  out_arr = func(
/home/b/babul/stlock/.virtualenvs/entropy_venv_12/lib/python3.11/site-packages/unyt/array.py:1949: RuntimeWarning: divide by zero encountered in power
  out_arr = func(


In [4]:
coords = get_coordinates_by_id(id_filepath, target_halo)
halo_center = unyt_array(coords, 'kpc')
halo_center/=1000
print(halo_center)

[105.932586 156.340703 106.44075 ] kpc


# Get Center of Gas Particles

In [5]:
# get center of gas particles 
gas_pos = data['gas', 'particle_position']

gas_pos_min = np.min(gas_pos, axis=0)
gas_pos_min.in_units('Mpc')

gas_pos_max = np.max(gas_pos, axis=0)
gas_pos_max.in_units('Mpc')

gas_zoom_centre = (gas_pos_min + gas_pos_max)/2.
gas_zoom_centre.in_units('Mpc')

gas_zoom_lengths = (gas_pos_max - gas_pos_min)/2.
gas_zoom_radius = np.mean(gas_zoom_lengths)
# gas_zoom_lengths.in_units('Mpc')
gas_zoom_radius.in_units('Mpc')

unyt_quantity(23.7859892, 'Mpc')

# Get minimum potential of gas particles

In [6]:
gas_pot = data['PartType0', 'Potential']
min_pot = np.min(gas_pot)
min_gas_pot_idx = np.argmin(gas_pot)
gas_pot[min_gas_pot_idx]
min_gas_pot_pos = data['gas', 'particle_position'][min_gas_pot_idx]
min_gas_pot_pos.in_units('Mpc')

max_gas_pot_idx = np.argmax(gas_pot)
max_gas_pot_pos = data['gas', 'position'][max_gas_pot_idx]
max_gas_pot_pos.in_units('Mpc')

min_pot_idx = np.argmin(data['all', 'Potential'])
min_pot_pos = data['all', 'particle_position'][min_pot_idx]
min_pot_pos.in_units('Mpc')

max_pot_idx = np.argmax(data['all', 'Potential'])
max_pot_pos = data['all', 'particle_position'][max_pot_idx]
max_pot_pos.in_units('Mpc')

unyt_array([122.38470818, 222.09494485, 193.81700368], 'Mpc')

# Get position of most massive black hole

In [7]:
# index
most_massive_bh_index = np.argmax(data['PartType5', 'BH_Mass'])
# position
most_massive_bh_pos = data['PartType5', 'particle_position'][most_massive_bh_index]
most_massive_bh_pos.in_units('Mpc')


most_dyn_massive_bh_index = np.argmax(data['PartType5', 'Masses'])

most_dyn_massive_bh_pos = data['PartType5', 'particle_position'][most_dyn_massive_bh_index]

most_dyn_massive_bh_pos.in_units('Mpc')

unyt_array([147.70096507, 154.55804228, 129.65819164], 'Mpc')

# Get position of highest gas density

In [8]:
rho_max_index = np.argmax(data['gas', 'density'])

rho_max_pos = data['gas', 'position'][rho_max_index]
rho_max_pos.in_units('Mpc')

rho_max_part_pos = data['gas', 'particle_position'][rho_max_index]
rho_max_part_pos.in_units('Mpc')

unyt_array([157.27002528, 164.43716682, 121.3396829 ], 'Mpc')

# Isolate IGRM

In [9]:
igrm = data.include_above(('PartType0', 'temperature'), 5e5, units='K')
igrm = igrm.include_below(('PartType0', 'H_nuclei_density'), 0.13, units='cm**-3')
igrm = igrm.include_equal(('PartType0', 'DelayTime'), 0)

## Calculate profile manually

In [10]:
# Radial Binning

radial_bins = unyt.unyt_array(np.append([0], np.logspace(-1, np.log10(5), 20)), 'Mpc')

halo_center.in_units('Mpc')
gas_zoom_centre.in_units('Mpc')
"""min_gas_pot_pos.in_units('Mpc')
max_gas_pot_pos.in_units('Mpc')
min_pot_pos.in_units('Mpc')
max_pot_pos.in_units('Mpc')
most_massive_bh_pos.in_units('Mpc')
most_dyn_massive_bh_pos.in_units('Mpc')"""
rho_max_pos.in_units('Mpc')

igrm_pos_x = igrm['gas', 'particle_position_x']
igrm_pos_y = igrm['gas', 'particle_position_y']
igrm_pos_z = igrm['gas', 'particle_position_z']



igrm_pos_x = igrm_pos_x.in_units('Mpc')
igrm_pos_y = igrm_pos_y.in_units('Mpc') 
igrm_pos_z = igrm_pos_z.in_units('Mpc')

In [11]:
print(igrm_pos_x)
print(igrm_pos_y)
print(igrm_pos_z)

print(halo_center.in_units('Mpc'))

[145.87320772 145.85628447 145.84431296 ... 147.8979205  147.05924862
 143.16055836] Mpc
[143.49080882 143.49588695 143.48671875 ... 132.6396829  132.31744026
 141.7849954 ] Mpc
[139.70847886 139.67537914 139.68353631 ... 129.90677849 130.42319623
 150.50785846] Mpc
[0.10593259 0.1563407  0.10644075] Mpc


In [12]:
igrm_halo_center = ((igrm_pos_x-halo_center[0])**2 + (igrm_pos_y-halo_center[1])**2 + 
                  (igrm_pos_z-halo_center[2])**2)**0.5


igrm_zoom_radii_rel_centre = ((igrm_pos_x-gas_zoom_centre[0])**2 + 
                  (igrm_pos_y-gas_zoom_centre[1])**2 + 
                  (igrm_pos_z-gas_zoom_centre[2])**2)**0.5

"""igrm_zoom_radii_rel_min_gas_pot = ((igrm_pos_x-min_gas_pot_pos[0])**2 + 
                  (igrm_pos_y-min_gas_pot_pos[1])**2 + 
                  (igrm_pos_z-min_gas_pot_pos[2])**2)**0.5

igrm_zoom_radii_rel_max_gas_pot = ((igrm_pos_x-max_gas_pot_pos[0])**2 + 
                  (igrm_pos_y-max_gas_pot_pos[1])**2 + 
                  (igrm_pos_z-max_gas_pot_pos[2])**2)**0.5

igrm_zoom_radii_rel_min_pot = ((igrm_pos_x-min_pot_pos[0])**2 + 
                  (igrm_pos_y-min_pot_pos[1])**2 + 
                  (igrm_pos_z-min_pot_pos[2])**2)**0.5

igrm_zoom_radii_rel_max_pot = ((igrm_pos_x-max_pot_pos[0])**2 + 
                  (igrm_pos_y-max_pot_pos[1])**2 + 
                  (igrm_pos_z-max_pot_pos[2])**2)**0.5

igrm_zoom_radii_rel_bh_mass_max = ((igrm_pos_x-most_massive_bh_pos[0])**2 + 
                  (igrm_pos_y-most_massive_bh_pos[1])**2 + 
                  (igrm_pos_z-most_massive_bh_pos[2])**2)**0.5

igrm_zoom_radii_rel_bh_dyn_mass_max = ((igrm_pos_x-most_dyn_massive_bh_pos[0])**2 + 
                  (igrm_pos_y-most_dyn_massive_bh_pos[1])**2 + 
                  (igrm_pos_z-most_dyn_massive_bh_pos[2])**2)**0.5"""

igrm_zoom_radii_rel_rho_max = ((igrm_pos_x-rho_max_pos[0])**2 + 
                  (igrm_pos_y-rho_max_pos[1])**2 + 
                  (igrm_pos_z-rho_max_pos[2])**2)**0.5

In [13]:
print(igrm_halo_center)
print(igrm_zoom_radii_rel_centre)
print(igrm_zoom_radii_rel_rho_max)

print(igrm_halo_center.in_units('Mpc'))
print(igrm_zoom_radii_rel_centre.in_units('Mpc'))
print(igrm_zoom_radii_rel_rho_max.in_units('Mpc'))

[247.55120136 247.52551213 247.51775368 ... 237.15487625 236.73675272
 251.28447671] Mpc
[ 4.17476992  4.17406321  4.18978056 ... 16.79279896 16.83830093
 13.6467804 ] Mpc
[30.10066492 30.08335816 30.09925471 ... 34.23903413 34.90626031
 39.53454681] Mpc
[247.55120136 247.52551213 247.51775368 ... 237.15487625 236.73675272
 251.28447671] Mpc
[ 4.17476992  4.17406321  4.18978056 ... 16.79279896 16.83830093
 13.6467804 ] Mpc
[30.10066492 30.08335816 30.09925471 ... 34.23903413 34.90626031
 39.53454681] Mpc


In [14]:
igrm_zoom_entropies = igrm['gas', 'entropy']
igrm_zoom_masses = igrm['gas', 'mass']

# change igrm_zoom_radii = .... to use the center you want!
igrm_zoom_radii = igrm_halo_center
igrm_zoom_radii.in_units('Mpc')

unyt_array([247.55120136, 247.52551213, 247.51775368, ..., 237.15487625,
       236.73675272, 251.28447671], 'Mpc')

In [15]:
print(igrm_halo_center)

[247.55120136 247.52551213 247.51775368 ... 237.15487625 236.73675272
 251.28447671] Mpc


In [16]:
entropy_profile_median = []
entropy_profile_mean = []
entropy_profile_mass_weighted = []
for bin_lo, bin_hi in zip(radial_bins[:-1], radial_bins[1:]):
#     print(bin_lo, bin_hi)
    radial_filter = np.logical_and(igrm_zoom_radii.in_units('Mpc').value>=bin_lo.in_units('Mpc').value,
                                   igrm_zoom_radii.in_units('Mpc').value<=bin_hi.in_units('Mpc').value)
#     print(np.where(radial_filter))
    entropies = igrm_zoom_entropies[radial_filter]
    masses = igrm_zoom_masses[radial_filter]
    # print(len(entropies))
    # print(len(masses))
#     print()
    
    entropy_profile_median.append(np.nanmedian(entropies))
    entropy_profile_mean.append(np.nanmean(entropies))
    if sum(masses)>0:
        K_mass = np.average(entropies, weights=masses)
        entropy_profile_mass_weighted.append(K_mass)
    else:
        entropy_profile_mass_weighted.append(np.nan)
    
    # print(K_mass)
    # print()

/home/b/babul/stlock/.virtualenvs/entropy_venv_12/lib/python3.11/site-packages/unyt/array.py:2040: RuntimeWarning: Mean of empty slice
  return func._implementation(*args, **kwargs)


In [17]:
radial_bin_centres = (radial_bins[:-1] + radial_bins[1:])/2.
radial_bin_centres_logged = np.log10(radial_bin_centres)
log_radial_bin_centres = (np.log10(radial_bins[:-1]) + np.log10(radial_bins[1:]))/2.
log_radial_bin_centres_unlogged = 10**log_radial_bin_centres

plt.plot(radial_bin_centres, entropy_profile_median, label='median', lw=3)
plt.plot(radial_bin_centres, entropy_profile_mean, label='mass-weighted', lw=3)
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('R [Mpc]')
plt.ylabel(r'K [keV $\mathrm{cm^2}$]')
plt.legend()

/home/b/babul/stlock/.virtualenvs/entropy_venv_12/lib/python3.11/site-packages/unyt/array.py:1824: RuntimeWarning: divide by zero encountered in log10
  out_arr = func(np.asarray(inp), out=out_func, **kwargs)


Error in callback <function _draw_all_if_interactive at 0x7f0abc7171a0> (for post_execute):


ValueError: Data has no positive values, and therefore cannot be log-scaled.

ValueError: Data has no positive values, and therefore cannot be log-scaled.

<Figure size 640x480 with 1 Axes>